# VinDr-Mammo: **Breast Density 4-Class** Classification

- **Target**: `breast_density` (bukan `breast_birads`)
- **Label mapping**: `DENSITY A`→0, `DENSITY B`→1, `DENSITY C`→2, `DENSITY D`→3
- **Loss**: BCEWithLogitsLoss → **CrossEntropyLoss** dengan class weights
- **Output head**: `NUM_CLASSES=4`, output `[B, 4]` logits
- **Metrics**: AUC-OvR macro, Macro-F1, per-class report, Quadratic Weighted Kappa
- **Sampler**: WeightedRandomSampler berbasis frekuensi 4 kelas
- **Dataset**: label `torch.long` untuk CrossEntropyLoss


## 1. Install & Import

In [ ]:
!pip install timm -q
!pip install scikit-learn -q

In [ ]:
import os, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

import timm
from sklearn.metrics import (
    classification_report, roc_auc_score, cohen_kappa_score, confusion_matrix,
    f1_score, accuracy_score, recall_score, precision_score,
    roc_curve, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")

## 2. Configuration

In [ ]:
class CFG:
    DATA_DIR          = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/dataset_preprocessed/dataset_preprocessed'
    CSV_PATH          = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/breast-level_annotations.csv'
    OUTPUT_DIR        = '/kaggle/working/outputs'

    MODEL_NAME        = 'swin_tiny_patch4_window7_224'
    PRETRAINED_WEIGHT = '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/swin_tiny_patch4_window7_224.pth'
    IMG_SIZE          = 224
    NUM_CLASSES       = 4
    CLASS_NAMES       = ['Density-A', 'Density-B', 'Density-C', 'Density-D']
    DROPOUT           = 0.3

    EPOCHS            = 100
    BATCH_SIZE        = 16
    NUM_WORKERS       = 4

    WARMUP_EPOCHS     = 5
    LR_BACKBONE       = 1e-5
    LR_HEAD           = 1e-4
    WEIGHT_DECAY      = 1e-4
    PATIENCE          = 15

    LABEL_SMOOTHING   = 0.1    # regularisasi, bantu kelas minor
    USE_MULTI_GPU     = True

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device      : {DEVICE}")
print(f"NUM_CLASSES : {CFG.NUM_CLASSES}")
print(f"CLASS_NAMES : {CFG.CLASS_NAMES}")


## 3. Data

In [ ]:
df = pd.read_csv(CFG.CSV_PATH)

# ── Parse breast_density → label 0-3 ──────────────────────────────────────────
# Format di CSV: "DENSITY A", "DENSITY B", "DENSITY C", "DENSITY D"
DENSITY_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

def parse_density(val):
    if pd.isna(val): return None
    letter = str(val).strip().upper().split()[-1]  # ambil huruf terakhir
    return DENSITY_MAP.get(letter, None)

df['label'] = df['breast_density'].apply(parse_density)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# Distribusi kelas
print("Distribusi Breast Density:")
for i, name in enumerate(CFG.CLASS_NAMES):
    n = (df['label'] == i).sum()
    print(f"  {name} (label={i}): {n:>5} samples ({n/len(df)*100:.1f}%)")
print(f"  Total: {len(df)}")

def build_pairs(df):
    pairs = []
    for (sid, lat), grp in df.groupby(['study_id', 'laterality']):
        cc  = grp[grp['view_position'] == 'CC']
        mlo = grp[grp['view_position'] == 'MLO']
        if len(cc) == 0 or len(mlo) == 0: continue
        pairs.append({
            'study_id':     sid,
            'laterality':   lat,
            'cc_image_id':  cc.iloc[0]['image_id'],
            'mlo_image_id': mlo.iloc[0]['image_id'],
            'label':        cc.iloc[0]['label'],   # density label 0-3
            'split':        cc.iloc[0]['split']
        })
    return pd.DataFrame(pairs)

pairs_df = build_pairs(df)
train_df = pairs_df[pairs_df['split'] == 'training'].reset_index(drop=True)
val_df   = pairs_df[pairs_df['split'] == 'test'].reset_index(drop=True)

if len(val_df) == 0:
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(pairs_df, test_size=0.2,
                                        stratify=pairs_df['label'], random_state=42)
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df)} pairs | Val: {len(val_df)} pairs")
print(f"[SANITY] CC==MLO image_id: {(train_df['cc_image_id']==train_df['mlo_image_id']).sum()} (harus 0)")

# Distribusi per-split
for split_name, sdf in [('Train', train_df), ('Val', val_df)]:
    dist = ' | '.join(f"{CFG.CLASS_NAMES[i]}:{(sdf['label']==i).sum()}" for i in range(4))
    print(f"  {split_name}: {dist}")


## 4. Dataset & DataLoader

In [ ]:
def get_image_path(data_dir, study_id, image_id):
    base = Path(data_dir) / str(study_id)
    for ext in ['.png', '.jpg', '.jpeg']:
        p = base / f'{image_id}{ext}'
        if p.exists(): return str(p)
    for ext in ['.png', '.jpg', '.jpeg']:
        p = Path(data_dir) / f'{image_id}{ext}'
        if p.exists(): return str(p)
    return None


def get_transforms(split='train', img_size=224):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if split == 'train':
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.2),
            T.RandomRotation(degrees=15),
            T.ColorJitter(brightness=0.3, contrast=0.3),
            T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ])
    return T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(mean, std)])


class VinDrDataset(Dataset):
    def __init__(self, df, data_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def _load(self, study_id, image_id):
        path = get_image_path(self.data_dir, study_id, image_id)
        if path is None:
            return Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
        return Image.open(path).convert('RGB')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cc  = self._load(row['study_id'], row['cc_image_id'])
        mlo = self._load(row['study_id'], row['mlo_image_id'])
        if self.transform:
            cc  = self.transform(cc)
            mlo = self.transform(mlo)
        return cc, mlo, torch.tensor(row['label'], dtype=torch.long)


train_dataset = VinDrDataset(train_df, CFG.DATA_DIR, get_transforms('train', CFG.IMG_SIZE))
val_dataset   = VinDrDataset(val_df,   CFG.DATA_DIR, get_transforms('val',   CFG.IMG_SIZE))

# ── Sampler: target proporsi = sqrt(count) / sum(sqrt(count)) ─────────────────
# Contoh hasil (B=16): A~1, B~4, C~9, D~5
# C tetap dominan (proporsional), tapi A dijamin minimal muncul ~1x/batch
labels_list  = train_df['label'].tolist()
class_counts = [labels_list.count(i) for i in range(CFG.NUM_CLASSES)]
sqrt_counts  = [math.sqrt(c) for c in class_counts]
total_sqrt   = sum(sqrt_counts)
target_dist  = [s / total_sqrt for s in sqrt_counts]   # target frekuensi per kelas

# weight per sample = target_dist[kelas] / jumlah_sample_kelas_itu
sample_weights = [target_dist[l] / class_counts[l] for l in labels_list]

sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=CFG.BATCH_SIZE,
    sampler=sampler, num_workers=CFG.NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Class counts (train): { {CFG.CLASS_NAMES[i]: class_counts[i] for i in range(4)} }")
print(f"Target dist/batch(B={CFG.BATCH_SIZE}): { {CFG.CLASS_NAMES[i]: f'{target_dist[i]*CFG.BATCH_SIZE:.1f}' for i in range(4)} }")

cc_b, mlo_b, lbl_b = next(iter(train_loader))
from collections import Counter
cnt = Counter(lbl_b.tolist())
print(f"Actual batch sample : { {CFG.CLASS_NAMES[k]: cnt.get(k,0) for k in range(4)} }")
print(f"CC == MLO: {torch.allclose(cc_b, mlo_b)} ← harus False")


## 5. Model — Late Fusion


In [ ]:
class SwinDensity_LateFusion(nn.Module):
    """
    Late-Fusion Swin Transformer untuk Breast Density 4-Class Classification.

    Architecture:
      CC  ──► Swin Encoder (shared weights) ──► cc_feat  [B, D]
      MLO ──► Swin Encoder (shared weights) ──► mlo_feat [B, D]
                                                     │
                                          torch.cat([cc_feat, mlo_feat], dim=1)  → [B, 2D]
                                                     │
                                               MLP Classifier
                                                     │
                                            logits [B, NUM_CLASSES]
    """

    def __init__(self, cfg):
        super().__init__()
        self.encoder = timm.create_model(
            cfg.MODEL_NAME, pretrained=True, num_classes=0, global_pool='avg'
        )
        D = self.encoder.num_features   # 768 untuk swin_tiny
        print(f"Encoder dim (D)        : {D}")
        print(f"Classifier input dim   : {D * 2}  (2×D)")

        self.classifier = nn.Sequential(
            nn.Linear(D * 2, 512),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT / 2),
            nn.Linear(128, cfg.NUM_CLASSES)   # 4 output
        )

        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, cc: torch.Tensor, mlo: torch.Tensor) -> torch.Tensor:
        cc_feat  = self.encoder(cc)
        mlo_feat = self.encoder(mlo)
        fused    = torch.cat([cc_feat, mlo_feat], dim=1)
        return self.classifier(fused)          # [B, 4] — NO squeeze


# ── Instantiate ────────────────────────────────────────────────────────────────
model = SwinDensity_LateFusion(CFG)

# ── Load pretrained weights (encoder only) ────────────────────────────────────
ckpt = torch.load(CFG.PRETRAINED_WEIGHT, map_location='cpu')
sd   = ckpt.get('model', ckpt.get('state_dict', ckpt)) if isinstance(ckpt, dict) else ckpt
ms   = model.encoder.state_dict()
filt = {k: v for k, v in sd.items() if k in ms and v.shape == ms[k].shape}
model.encoder.load_state_dict(filt, strict=False)
print(f"Pretrained: {len(filt)}/{len(ms)} layers loaded")
assert len(filt) > 100, f"Terlalu sedikit layer loaded ({len(filt)}) — cek file .pth!"

if CFG.USE_MULTI_GPU and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(DEVICE)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")


## 6. Loss & Optimizer

In [ ]:
# ── Focal Loss: secara otomatis fokus ke sampel yang susah dipelajari ─────────
# γ (gamma) mengontrol fokus: γ=0 → sama dengan CrossEntropy
# γ=2 → sangat fokus ke hard samples (kelas yang sering salah)
# α (class weight) tetap pakai log-scale tapi lebih ringan

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        # Cross entropy per sample (tidak di-reduce dulu)
        ce = F.cross_entropy(logits, targets,
                             weight=self.weight,
                             label_smoothing=self.label_smoothing,
                             reduction='none')
        # p_t = exp(-ce) → probabilitas prediksi benar
        pt      = torch.exp(-ce)
        focal_w = (1 - pt) ** self.gamma
        return (focal_w * ce).mean()


labels_list  = train_df['label'].tolist()
class_counts = [labels_list.count(i) for i in range(CFG.NUM_CLASSES)]
max_c        = max(class_counts)
log_w        = [1.0 + math.log(max_c / c) for c in class_counts]
mean_w       = sum(log_w) / len(log_w)
norm_w       = [w / mean_w for w in log_w]

class_w   = torch.tensor(norm_w, dtype=torch.float32).to(DEVICE)
criterion = FocalLoss(weight=class_w, gamma=2.0, label_smoothing=CFG.LABEL_SMOOTHING)

print(f"Loss: FocalLoss(gamma=2.0) + label_smoothing={CFG.LABEL_SMOOTHING}")
print(f"Class counts : { {CFG.CLASS_NAMES[i]: class_counts[i] for i in range(4)} }")
print(f"Log-scale w  : { {CFG.CLASS_NAMES[i]: f'{norm_w[i]:.3f}' for i in range(4)} }")


def get_optimizer(model, cfg, backbone_frozen=True):
    base = model.module if hasattr(model, 'module') else model
    backbone_lr = 0.0 if backbone_frozen else cfg.LR_BACKBONE
    return torch.optim.AdamW([
        {'params': base.encoder.parameters(),    'lr': backbone_lr,  'weight_decay': cfg.WEIGHT_DECAY},
        {'params': base.classifier.parameters(), 'lr': cfg.LR_HEAD,  'weight_decay': cfg.WEIGHT_DECAY},
    ])


def freeze_backbone(model, freeze=True):
    base = model.module if hasattr(model, 'module') else model
    for p in base.encoder.parameters():
        p.requires_grad = not freeze
    print(f"Backbone {'FROZEN' if freeze else 'UNFROZEN'}")


freeze_backbone(model, freeze=True)
optimizer = get_optimizer(model, CFG, backbone_frozen=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.EPOCHS, eta_min=1e-7
)
print(f"LR head: {CFG.LR_HEAD} | LR backbone: {CFG.LR_BACKBONE} (frozen {CFG.WARMUP_EPOCHS} epoch pertama)")


## 7. Diagnostic Pre-Training

In [ ]:
print("=" * 60)
print("DIAGNOSTIC v8 — Breast Density 4-Class")
print("=" * 60)

cc_b, mlo_b, lbl_b = next(iter(train_loader))
cc_b, mlo_b, lbl_b = cc_b.to(DEVICE), mlo_b.to(DEVICE), lbl_b.to(DEVICE)

# [1] CC vs MLO pixel check
identical = torch.allclose(cc_b, mlo_b)
print(f"\n[1] CC == MLO pixel-identik: {identical}", '❌ BUG!' if identical else '✅ OK')

# [2] Feature & output shape check
base = model.module if hasattr(model, 'module') else model
base.eval()
with torch.no_grad():
    logits = base(cc_b, mlo_b)
print(f"[2] Output shape: {logits.shape}  ← harus [B, 4]", '✅' if logits.shape[1] == 4 else '❌')

# [3] Label range check
print(f"[3] Label range: min={lbl_b.min().item()}, max={lbl_b.max().item()}  ← harus 0-3")
print(f"[4] Label dtype: {lbl_b.dtype}  ← harus torch.int64")

base.train()
print("\n✅ Diagnostic selesai")


## 8. Training

In [ ]:
def _safe_auc(labels, probs):
    """AUC-OvR macro. Normalize probs supaya sum=1 (fix float precision issue)."""
    try:
        # Normalize tiap baris supaya sum persis = 1.0
        probs_norm = probs / probs.sum(axis=1, keepdims=True)
        if len(np.unique(labels)) < CFG.NUM_CLASSES:
            missing = set(range(CFG.NUM_CLASSES)) - set(np.unique(labels).tolist())
            print(f"  [AUC] Kelas tidak muncul: {[CFG.CLASS_NAMES[i] for i in missing]} → AUC=0.0")
            return 0.0
        return roc_auc_score(labels, probs_norm, multi_class='ovr', average='macro')
    except Exception as e:
        print(f"  [AUC error] {e}")
        return 0.0


def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    all_preds, all_probs, all_labels = [], [], []

    for cc, mlo, labels in tqdm(loader, desc='Train', leave=False):
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(cc, mlo)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        probs = torch.softmax(logits.float(), dim=1).detach().cpu().numpy()
        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    all_probs  = np.array(all_probs,  dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.int64)
    all_preds  = np.array(all_preds,  dtype=np.int64)

    auc = _safe_auc(all_labels, all_probs)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), auc, f1, acc


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_probs, all_labels = [], [], []

    for cc, mlo, labels in tqdm(loader, desc='Val', leave=False):
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            logits = model(cc, mlo)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        probs = torch.softmax(logits.float(), dim=1).detach().cpu().numpy()
        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    all_probs  = np.array(all_probs,  dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.int64)
    all_preds  = np.array(all_preds,  dtype=np.int64)

    auc = _safe_auc(all_labels, all_probs)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)

    # Per-class accuracy untuk monitoring collapse
    per_cls = []
    for i in range(CFG.NUM_CLASSES):
        mask = all_labels == i
        if mask.sum() > 0:
            cls_acc = accuracy_score(all_labels[mask], all_preds[mask])
            per_cls.append(f"{CFG.CLASS_NAMES[i]}:{cls_acc:.2f}")
    print(f"  Per-class acc: {' | '.join(per_cls)}")

    return total_loss / len(loader), auc, f1, acc, all_probs, all_labels, all_preds


print("Training functions defined.")


In [ ]:
scaler  = torch.cuda.amp.GradScaler()
history = {k: [] for k in ['tr_loss', 'vl_loss', 'tr_auc', 'vl_auc', 'tr_f1', 'vl_f1']}

best_auc        = 0.0
patience_count  = 0
best_probs      = None
best_labels_arr = None
best_preds_arr  = None

print(f"{'Ep':>3} | {'TrLoss':>7} | {'TrAUC':>6} | {'TrF1':>6} | {'VlAUC':>6} | {'VlF1':>6} | {'VlAcc':>6} | Status")
print('-' * 85)

for epoch in range(1, CFG.EPOCHS + 1):

    if epoch == CFG.WARMUP_EPOCHS + 1:
        freeze_backbone(model, freeze=False)
        optimizer = get_optimizer(model, CFG, backbone_frozen=False)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.EPOCHS - CFG.WARMUP_EPOCHS, eta_min=1e-7
        )
        print(f"  → Backbone unfrozen @ epoch {epoch}")

    tr_loss, tr_auc, tr_f1, tr_acc = train_epoch(
        model, train_loader, optimizer, criterion, DEVICE, scaler
    )
    vl_loss, vl_auc, vl_f1, vl_acc, vl_probs, vl_labels, vl_preds = val_epoch(
        model, val_loader, criterion, DEVICE
    )
    scheduler.step()

    history['tr_loss'].append(tr_loss); history['vl_loss'].append(vl_loss)
    history['tr_auc'].append(tr_auc);   history['vl_auc'].append(vl_auc)
    history['tr_f1'].append(tr_f1);     history['vl_f1'].append(vl_f1)

    status = ''
    if vl_auc > best_auc:
        best_auc = vl_auc
        patience_count = 0
        base = model.module if hasattr(model, 'module') else model
        torch.save(base.state_dict(), f'{CFG.OUTPUT_DIR}/best_model.pth')
        best_probs, best_labels_arr, best_preds_arr = vl_probs, vl_labels, vl_preds
        status = '✓ SAVED'
    else:
        patience_count += 1
        if patience_count >= CFG.PATIENCE:
            print(f"\nEarly stopping @ epoch {epoch}")
            break

    print(f"{epoch:>3} | {tr_loss:>7.4f} | {tr_auc:>6.4f} | {tr_f1:>6.4f} | "
          f"{vl_auc:>6.4f} | {vl_f1:>6.4f} | {vl_acc:>6.4f} | {status}")

print(f"\n✅ Selesai. Best Val AUC (OvR macro): {best_auc:.4f}")


## 9. Evaluation

In [ ]:
base = model.module if hasattr(model, 'module') else model
base.load_state_dict(torch.load(f'{CFG.OUTPUT_DIR}/best_model.pth', map_location=DEVICE))

vl_loss, vl_auc, vl_f1, vl_acc, best_probs, best_labels_arr, best_preds_arr = val_epoch(
    model, val_loader, criterion, DEVICE
)

auc     = roc_auc_score(best_labels_arr, best_probs, multi_class='ovr', average='macro')
acc     = accuracy_score(best_labels_arr, best_preds_arr)
f1_mac  = f1_score(best_labels_arr, best_preds_arr, average='macro',    zero_division=0)
f1_wt   = f1_score(best_labels_arr, best_preds_arr, average='weighted', zero_division=0)
kappa   = cohen_kappa_score(best_labels_arr, best_preds_arr, weights='quadratic')

print("=" * 55)
print("     EVALUATION RESULTS — Breast Density 4-Class")
print("=" * 55)
print(f"  AUC-ROC (OvR macro) : {auc:.4f}")
print(f"  Accuracy            : {acc:.4f}")
print(f"  Macro F1            : {f1_mac:.4f}")
print(f"  Weighted F1         : {f1_wt:.4f}")
print(f"  Quadratic Kappa     : {kappa:.4f}")
print("=" * 55)
print(classification_report(best_labels_arr, best_preds_arr,
      target_names=CFG.CLASS_NAMES, zero_division=0))


## 10. Visualisasi

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history['tr_loss'], label='Train', color='#3498db', lw=2)
ax1.plot(history['vl_loss'], label='Val',   color='#e74c3c', lw=2)
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history['tr_auc'], label='Train', color='#3498db', lw=2)
ax2.plot(history['vl_auc'], label='Val',   color='#e74c3c', lw=2)
ax2.axhline(y=best_auc, color='green', ls='--', alpha=0.7, label=f'Best={best_auc:.4f}')
ax2.set_title('AUC-ROC (OvR macro)'); ax2.legend(); ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(history['tr_f1'], label='Train', color='#3498db', lw=2)
ax3.plot(history['vl_f1'], label='Val',   color='#e74c3c', lw=2)
ax3.set_title('Macro F1'); ax3.legend(); ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[1, 0])
cm = confusion_matrix(best_labels_arr, best_preds_arr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
            xticklabels=CFG.CLASS_NAMES, yticklabels=CFG.CLASS_NAMES)
ax4.set_title('Confusion Matrix')
ax4.set_xlabel('Predicted'); ax4.set_ylabel('True')
plt.setp(ax4.get_xticklabels(), rotation=30)

ax5 = fig.add_subplot(gs[1, 1])
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc as sk_auc
y_bin = label_binarize(best_labels_arr, classes=[0,1,2,3])
colors = ['#3498db','#e74c3c','#2ecc71','#f39c12']
for i, (name, col) in enumerate(zip(CFG.CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], best_probs[:, i])
    ax5.plot(fpr, tpr, color=col, lw=2, label=f'{name} (AUC={sk_auc(fpr,tpr):.3f})')
ax5.plot([0,1],[0,1],'k--',alpha=0.4)
ax5.set_title('ROC per Class'); ax5.legend(fontsize=8); ax5.grid(alpha=0.3)
ax5.set_xlabel('FPR'); ax5.set_ylabel('TPR')

ax6 = fig.add_subplot(gs[1, 2])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', ax=ax6,
            xticklabels=CFG.CLASS_NAMES, yticklabels=CFG.CLASS_NAMES)
ax6.set_title('Confusion Matrix (Normalized)')
ax6.set_xlabel('Predicted'); ax6.set_ylabel('True')
plt.setp(ax6.get_xticklabels(), rotation=30)

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/results_v8_density.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan.")


## 11. Summary

In [ ]:
print("="*60)
print("     FINAL SUMMARY — v8 Breast Density 4-Class")
print("="*60)
print(f"  Model      : {CFG.MODEL_NAME}")
print(f"  Task       : Breast Density (A/B/C/D)")
print(f"  LR         : backbone={CFG.LR_BACKBONE}, head={CFG.LR_HEAD}")
print(f"  Warmup     : {CFG.WARMUP_EPOCHS} epoch")
print("-"*60)
print(f"  AUC-ROC    : {auc:.4f}")
print(f"  Accuracy   : {acc:.4f}")
print(f"  Macro F1   : {f1_mac:.4f}")
print(f"  QW Kappa   : {kappa:.4f}")
print("="*60)

pd.DataFrame([{
    'auc_ovr_macro': auc, 'accuracy': acc,
    'macro_f1': f1_mac,   'weighted_f1': f1_wt,
    'quadratic_kappa': kappa,
}]).to_csv(f'{CFG.OUTPUT_DIR}/metrics_v8_density.csv', index=False)
print("Saved to metrics_v8_density.csv")
